# Constant-Potential SJM–NEB

## Goal of the notebook

Let us construct a hydrogen migration path on solvated Pt(111) while we hold the electrode potential consistent across the path.

## Description of the method

The solvated jellium method (SJM) in GPAW adjusts the electronic charge to reach a target potential in an implicit solvent environment. The nudged elastic band (NEB) method relaxes a sequence of images between initial and final adsorption geometries.

Each image has its own calculator and must meet compatible potential and force criteria. SJM uses its own potential reference; comparison with an experimental reference electrode requires an explicit conversion.

## What we will do

1. Inspect a solvated Pt slab and relax hydrogen at its initial and final adsorption sites.
2. Interpolate the path and configure separate SJM calculators for the Dynamic NEB images.
3. Inspect the relaxed path and extract an activation energy from the profile.


In [ ]:
# Install course dependencies only in Google Colab. Local runs use environment.yml.
!if python -c "import google.colab" >/dev/null 2>&1; then apt-get update -qq && apt-get install -y libxc-dev && pip install -q gpaw gdown py3Dmol ase matplotlib numpy && echo "y" | gpaw install-data /usr/local/pkgs/gpaw-data-0.9.20000/; fi

In [ ]:
import ase.io
import numpy as np
import os
import py3Dmol

from ase import Atom
from ase.build import fcc111, molecule
from ase.constraints import FixAtoms
from ase.io import read, write
from ase.io.cube import read_cube_data, write_cube
from ase.mep import DyNEB, interpolate, NEBTools
from ase.optimize import BFGS
from ase.units import m, Pascal
from ase.visualize import view
from gpaw import setup_paths
from gpaw.solvation import (EffectivePotentialCavity, GradientSurface, LinearDielectric, SurfaceInteraction)
from gpaw.solvation.sjm import SJM, SJMPower12Potential
from matplotlib import pyplot as plt

In [ ]:
# Standard publication figure sizes in inches
plt.rc("figure", figsize  = (8.25 / 2.54, 8.25 / 2.54))
plt.rc("figure", dpi      = 100)
plt.rc("lines",  linewidth= 4)
plt.rc("font",   size     = 10)
plt.rc("font",   family   = "sans-serif")
plt.rc("axes",   titlesize= 10)
plt.rc("axes",   labelsize= 9)
plt.rc("xtick",  labelsize= 8)
plt.rc("ytick",  labelsize= 8)
plt.rc("legend", fontsize = 8)
plt.rc("figure", titlesize= 10)

In [ ]:

setup_paths.insert(0, "/usr/local/pkgs/gpaw-data-0.9.20000-hd8ed1ab_2/share/gpaw/")

## Build the initial model

Let us build a Pt slab, place a water molecule above it, and calculate the system potential energy.


Inspect the optimized endpoint. Check the adsorbate site, solvent geometry, and constrained slab region before you continue.


In [ ]:
!gdown --fuzzy "https://drive.google.com/file/d/1JaonEpy0oLvKI8DDKOLfsoyjUd2o9IYT/view?usp=drive_link" -O Pt111-water.xyz
atoms = ase.io.read("Pt111-water.xyz")
view(atoms, viewer="x3d")

## Optimize the initial state

We load the pre-optimized solvated Pt(111) slab prepared for this calculation and place a hydrogen atom above an atop site. This structure defines our initial state. First, we use a loose potential tolerance with `always_adjust=True`, so ionic and electronic degrees of freedom can relax together. Then we tighten the tolerances for a well-converged structure. Plan about 1 hour and 20 minutes for this calculation.

> **Before you begin:** Record the target potential, convergence tolerances, and endpoint geometry.
>
> **What to look for:** Build the band after both endpoints converge at the same potential. This keeps an endpoint mismatch out of the barrier.


In [ ]:
atoms.append(Atom("H", atoms[2].position + (0.0, 0.0, 1.5)))
view(atoms, viewer="x3d")

About 1 hour and 20 minutes: relax the slab and adsorbate.

Let us add an H atom to the solvated Pt slab.


A joint geometry and potential optimization can save time. We set a loose potential tolerance and `always_adjust=True`. The calculation can then take ionic steps while it adjusts the electron count. The potential may vary during the early steps; the final iterations target a constant potential.


In [ ]:
atoms.set_constraint(FixAtoms(indices=[0, 1]))
view(atoms, viewer="x3d")
# Keep the electrode potential and implicit-solvent model consistent across endpoints.
sjm_parameters = {"target_potential": 4.2, "tol": 0.2, "always_adjust": True}
water_dielectric_constant = 78.36
surface_tension = 18.4 * 0.001 * Pascal * m
cavity = EffectivePotentialCavity(effective_potential=SJMPower12Potential(H2O_layer=False), temperature=298.15, surface_calculator=GradientSurface())
dielectric = LinearDielectric(epsinf=water_dielectric_constant)
interactions = [SurfaceInteraction(surface_tension=surface_tension)]
calculator = SJM(mode="lcao", txt="Pt111-H-sim.txt", gpts=(16, 16, 136), kpts=(2, 2, 1), xc="PBE", maxiter=1000, sj=sjm_parameters, cavity=cavity, dielectric=dielectric, interactions=interactions)
atoms.calc = calculator
optimizer = BFGS(atoms, trajectory="qn-Pt111-H-sim.traj", logfile="qn-Pt111-H-sim.log")
optimizer.run()
sjm_parameters["tol"] = 0.01
sjm_parameters["always_adjust"] = False
sjm_parameters["slope"] = None
calculator.set(sj=sjm_parameters)
optimizer = BFGS(atoms, trajectory="qn-Pt111-H-sim-1.traj", logfile="qn-Pt111-H-sim.log")
optimizer.run()

In [ ]:
!gdown --fuzzy "https://drive.google.com/file/d/1pTlT-frvB4wIBwG2ZsfDVHrs9BY0vASc/view?usp=drive_link" -O qn-Pt111-H-sim-1.traj
atoms = ase.io.read("qn-Pt111-H-sim-1.traj", index=-1)
view(atoms, viewer="x3d")

## Optimize the final state

Move H from the atop site to the target hollow site and keep a comparable starting height. This gives us the final endpoint for the diffusion path.


In [ ]:
hydrogen_height_angstrom = atoms[10].z
atoms[10].position = (atoms[1].position + atoms[2].position + atoms[3].position) / 3
atoms[10].x = atoms[0].x
atoms[10].z = hydrogen_height_angstrom
view(atoms, viewer="x3d")

In [ ]:
sjm_parameters = {"target_potential": 4.2, "tol": 0.2, "always_adjust": True, "excess_electrons": -0.01232, "slope": -50.0,}
water_dielectric_constant = 78.36
surface_tension = 18.4 * 0.001 * Pascal * m
cavity = EffectivePotentialCavity(effective_potential=SJMPower12Potential(H2O_layer=False), temperature=298.15, surface_calculator=GradientSurface())
dielectric = LinearDielectric(epsinf=water_dielectric_constant)
interactions = [SurfaceInteraction(surface_tension=surface_tension)]
calculator = SJM(mode="lcao", txt="Pt111-H-hollow.txt", gpts=(16, 16, 136), kpts=(2, 2, 1), xc="PBE", maxiter=1000, sj=sjm_parameters, cavity=cavity, dielectric=dielectric, interactions=interactions)
atoms.calc = calculator
optimizer = BFGS(atoms, trajectory="qn-Pt111-H-hollow.traj", logfile="qn-Pt111-H-hollow.log")
optimizer.run()
sjm_parameters["tol"] = 0.01
sjm_parameters["always_adjust"] = False
sjm_parameters["slope"] = None
calculator.set(sj=sjm_parameters)
optimizer = BFGS(atoms, trajectory="qn-Pt111-H-hollow-1.traj", logfile="qn-Pt111-H-hollow-1.log")
optimizer.run()

In [ ]:
!gdown --fuzzy "https://drive.google.com/file/d/1pTlT-frvB4wIBwG2ZsfDVHrs9BY0vASc/view?usp=drive_link" -O qn-Pt111-H-hollow-1.traj
atoms = ase.io.read("qn-Pt111-H-hollow-1.traj", index=-1)
view(atoms, viewer="x3d")

## Run the constant-potential NEB

The initial and final states now share a converged electrode potential. We interpolate five intermediate images, attach an independent SJM calculator to each image, and optimize the band with Dynamic NEB (DyNEB). Plan for more than 12 hours. Save intermediate trajectories so you can resume the calculation and diagnose its progress.

> **Pause and predict:** Sketch the barrier profile for diffusion from atop to hollow.
>
> **What to look for:** Inspect each image for a discontinuity before you accept the fitted barrier.


In [ ]:
initial = ase.io.read("qn-Pt111-H-sim-1.traj")
final = ase.io.read("qn-Pt111-H-hollow-1.traj")
images = [initial]
for index in range(5):
    image = initial.copy()
    # Each NEB image needs its own calculator and solvent objects.
    sjm_parameters = {"target_potential": 4.2, "tol": 0.01, "excess_electrons": -0.01232, "slope": -50.0,}
    water_dielectric_constant = 78.36
    surface_tension = 18.4 * 0.001 * Pascal * m
    cavity = EffectivePotentialCavity(effective_potential=SJMPower12Potential(H2O_layer=False), temperature=298.15, surface_calculator=GradientSurface())
    dielectric = LinearDielectric(epsinf=water_dielectric_constant)
    interactions = [SurfaceInteraction(surface_tension=surface_tension)]
    calculator = SJM(mode="lcao", txt=f"gpaw-{index + 1:d}.txt", gpts=(16, 16, 136), kpts=(2, 2, 1), xc="PBE", maxiter=1000, sj=sjm_parameters, cavity=cavity, dielectric=dielectric, interactions=interactions)
    image.calc = calculator
    images.append(image)
images.append(final)
interpolate(images)
reaction_path = DyNEB(images)
optimizer = BFGS(reaction_path, logfile="neb.log", trajectory="neb.traj")
optimizer.run(fmax=0.05)

## Analyze the activation barrier

After NEB convergence, we extract the final images and use `NEBTools` to plot energy along the minimum-energy path. The highest-energy image gives the activation barrier.


In [ ]:
!gdown --fuzzy "https://drive.google.com/file/d/1pTlT-frvB4wIBwG2ZsfDVHrs9BY0vASc/view?usp=drive_link" -O neb.traj
atoms = ase.io.read("neb.traj", index=-1)
view(atoms, viewer="x3d")

In [ ]:

images = read("neb.traj@-7:")
print(len(images))

In [ ]:
images = ase.io.read("neb.traj", index=":-3")
neb_tools = NEBTools(images)
figure = neb_tools.plot_band()
plt.show()
figure.savefig("band.png")

## What have you learned

Let us gather the main ideas from this notebook. You can now:

- Set up endpoint optimizations at one target potential.
- Explain how adjustable charge, solvent parameters, and convergence tolerances shape the calculation.
- Construct a Dynamic NEB calculation with an independent calculator for each image.
- Read a migration barrier and identify convergence or path issues that affect it.

### Questions for discussion

1. How could endpoints at different electrode potentials distort the apparent barrier?
2. Which checks can distinguish a physical path maximum from an unconverged or discontinuous image?
